# 06 — Improving Unseen-Acronym Generalization

Builds on `04_main` (rank-16 QLoRA, greedy decoding) and tests three changes aimed at **unseen** exact match. Each change has a flag, so the notebook can run them together or one at a time as ablations:

| Flag | Idea | What changes |
| --- | --- | --- |
| `DIVERSE_TRAIN` | Acronym-diverse training | The 1,000 training examples come from many ambiguous acronyms (capped per acronym) instead of only the 30 seen ones. The 10 unseen test acronyms are never trained on. |
| `REGULARIZED` | Less overfitting to seen labels | Lower LR, higher LoRA dropout, weight decay, NEFTune, and early stopping on the loss of a **held-out dev set of different acronyms**. |
| `CONTEXT_MODE` | More context | `"full"` gives the whole abstract; `"local60"` keeps the ±60-word window used in 04. |

**Decoding** (not a flag; every run reports all three from the same trained adapter): greedy, as in 04; plain beam search (top-1 of `NUM_BEAMS` beams); and **beam + initials re-ranking**, which picks the highest-scoring beam whose letters are consistent with the acronym (Schwartz & Hearst check, e.g. `PD` → "power doppler" passes, `PD` → "peripheral nerve" fails) and falls back to the top beam if none pass.

Both test sets (500 seen, 500 unseen) come from exactly the same split code as notebooks 01–05, so the results are directly comparable. Prompts are never truncated by tokens: if one is too long, its word window shrinks so the `Expansion:` line and the assistant header always survive.

In [ ]:
# 1. Install Dependencies (Compatible with Colab T4 & Zero Drive Writes)
!pip install -q "pyarrow<24,>=19.0.0" transformers datasets accelerate peft bitsandbytes scikit-learn

import os
import re
import time
from collections import Counter
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
# 2. Mount Google Drive (Read-Only Access to valid.csv)
drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/valid.csv"
assert os.path.exists(CSV_PATH), f"File not found at {CSV_PATH}. Check your Drive folder path!"

In [ ]:
# 3. Configuration — the three flags select which improvements are on
DIVERSE_TRAIN = True      # Idea 2: train on many acronyms instead of only the 30 seen ones
REGULARIZED   = True      # Idea 5: lower LR, more dropout, NEFTune, early stopping on held-out acronyms
CONTEXT_MODE  = "full"    # Idea 7: "full" = whole abstract, "local60" = ±60 words (as in 04)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
SEED = 42
N_TRAIN = 1000            # Same training budget as notebooks 02–05
MAX_PER_ACRONYM = 8       # Diverse mode: at most this many training rows per acronym
N_DEV_ACRONYMS = 10       # Held-out acronyms (neither seen nor unseen-test) for early stopping
N_DEV = 300

LOCAL_WINDOW = 60
MAX_LENGTH = 1024 if CONTEXT_MODE == "full" else 448   # Full abstracts reach ~900 prompt tokens
TARGET_BUDGET = 32
PROMPT_BUDGET = MAX_LENGTH - TARGET_BUDGET

BATCH_SIZE = 4            # Smaller per-device batch for long prompts (Llama's 128k-vocab logits are large)
GRAD_ACCUM = 4            # Effective batch size = 16, same as 04

LORA_R, LORA_ALPHA = 16, 32
if REGULARIZED:
    LEARNING_RATE, LORA_DROPOUT, WEIGHT_DECAY = 1e-4, 0.10, 0.01
    NEFTUNE_ALPHA = 5.0
    MAX_EPOCHS = 4        # Upper bound; early stopping on dev loss usually ends sooner
    EVAL_STEPS, PATIENCE = 20, 3
else:                     # Exactly the 04 settings
    LEARNING_RATE, LORA_DROPOUT, WEIGHT_DECAY = 2e-4, 0.05, 0.0
    NEFTUNE_ALPHA = None
    MAX_EPOCHS = 3

NUM_BEAMS = 5             # Idea 3: beam width; all beams are kept as candidates for initials re-ranking

RUN_TAG = f"diverse={DIVERSE_TRAIN} | regularized={REGULARIZED} | context={CONTEXT_MODE}"
RUN_SLUG = f"d{int(DIVERSE_TRAIN)}_r{int(REGULARIZED)}_{CONTEXT_MODE}"
print(RUN_TAG)

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# 4. Exact Same In-Memory Split as Notebooks 01–05 (test sets are identical)
raw_df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
raw_df["LOCATION"] = raw_df["LOCATION"].astype(int)

def get_acronym(row):
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    return words[loc] if 0 <= loc < len(words) else ""

raw_df["ACRONYM"] = raw_df.apply(get_acronym, axis=1)
raw_df = raw_df[raw_df["ACRONYM"] != ""]

# Filter to truly ambiguous acronyms (>= 2 competing expansions)
ambig_counts = raw_df.groupby("ACRONYM")["LABEL"].nunique()
ambiguous_acronyms = ambig_counts[ambig_counts >= 2].index
ambig_df = raw_df[raw_df["ACRONYM"].isin(ambiguous_acronyms)].copy()

label_counts = ambig_df["LABEL"].value_counts()
valid_labels = label_counts[label_counts >= 30].index
ambig_df = ambig_df[ambig_df["LABEL"].isin(valid_labels)].copy()

top_ambig_acronyms = (
    ambig_df.groupby("ACRONYM")
    .filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"]
    .value_counts()
    .head(40)
    .index.tolist()
)

np.random.seed(SEED)
np.random.shuffle(top_ambig_acronyms)
seen_acronyms = set(top_ambig_acronyms[:30])
unseen_acronyms = set(top_ambig_acronyms[30:])

seen_pool = ambig_df[ambig_df["ACRONYM"].isin(seen_acronyms)].copy()
unseen_pool = ambig_df[ambig_df["ACRONYM"].isin(unseen_acronyms)].copy()

sampled_seen, _ = train_test_split(
    seen_pool,
    train_size=1500,
    stratify=seen_pool["LABEL"],
    random_state=SEED,
)
seen_train_df, test_seen_df = train_test_split(
    sampled_seen,
    train_size=1000,
    test_size=500,
    stratify=sampled_seen["LABEL"],
    random_state=SEED,
)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED).copy()

print(f"Shared split -> Seen-acronym train: {len(seen_train_df)} | Seen Test: {len(test_seen_df)} | Unseen Test: {len(test_unseen_df)}")

In [ ]:
# 5. Held-Out Dev Acronyms + (Optionally) Acronym-Diverse Training Set
# Rows that may never be trained on: every test row, plus any row sharing an abstract with a test row
test_texts = set(test_seen_df["TEXT"]) | set(test_unseen_df["TEXT"])

# Every ambiguous acronym that still has >= 2 senses after the label filter
multi_sense = ambig_df.groupby("ACRONYM")["LABEL"].nunique()
multi_sense = set(multi_sense[multi_sense >= 2].index)

# Dev acronyms: neither seen nor unseen-test, never trained on. Their loss acts as a
# stand-in for unseen-acronym performance when choosing when to stop training.
rng = np.random.RandomState(SEED)
dev_candidates = sorted(multi_sense - set(top_ambig_acronyms))
dev_acronyms = {str(a) for a in rng.choice(dev_candidates, size=min(N_DEV_ACRONYMS, len(dev_candidates)), replace=False)}
dev_pool = ambig_df[ambig_df["ACRONYM"].isin(dev_acronyms) & ~ambig_df["TEXT"].isin(test_texts)]
dev_df = dev_pool.sample(n=min(N_DEV, len(dev_pool)), random_state=SEED).copy()

if DIVERSE_TRAIN:
    pool = ambig_df[
        ambig_df["ACRONYM"].isin(multi_sense)
        & ~ambig_df["ACRONYM"].isin(unseen_acronyms | dev_acronyms)
        & ~ambig_df["TEXT"].isin(test_texts | set(dev_df["TEXT"]))
    ].sample(frac=1.0, random_state=SEED)
    # Round-robin over each acronym's senses, then keep at most MAX_PER_ACRONYM rows per acronym
    pool["_sense_rank"] = pool.groupby(["ACRONYM", "LABEL"]).cumcount()
    pool = pool.sort_values(["_sense_rank"], kind="stable")
    capped = pool.groupby("ACRONYM", sort=False).head(MAX_PER_ACRONYM)
    if len(capped) >= N_TRAIN:
        train_df = capped.sample(n=N_TRAIN, random_state=SEED)
    else:  # Fewer acronyms than expected: top up from the rest of the pool
        rest = pool.drop(capped.index)
        train_df = pd.concat([capped, rest.sample(n=min(N_TRAIN - len(capped), len(rest)), random_state=SEED)])
    train_df = train_df.drop(columns="_sense_rank").sample(frac=1.0, random_state=SEED)
else:
    train_df = seen_train_df.copy()

assert not set(train_df["ACRONYM"]) & unseen_acronyms, "Unseen acronyms leaked into training"
assert not set(train_df["ACRONYM"]) & dev_acronyms, "Dev acronyms leaked into training"
assert not set(train_df["TEXT"]) & test_texts, "Test abstracts leaked into training"

print(f"Train: {len(train_df)} rows | {train_df['ACRONYM'].nunique()} acronyms | {train_df['LABEL'].nunique()} senses")
print(f"  of which from the 30 seen acronyms: {train_df['ACRONYM'].isin(seen_acronyms).sum()} rows")
print(f"Dev (early stopping): {len(dev_df)} rows | acronyms: {sorted(dev_acronyms)}")
print(f"Unseen test acronyms: {sorted(unseen_acronyms)}")

In [ ]:
# 6. Prompts — never truncated by tokens; long contexts shrink their word window instead
SYSTEM_PROMPT = (
    "You are a clinical and biomedical text parser. Expand the bracketed abbreviation "
    "based on the context. Output ONLY the exact expanded term in lowercase, with no "
    "punctuation, quotes, or explanation."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(context_str, abbr):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Abbreviation: [{abbr}]\nContext: {context_str}\nExpansion:"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def n_tokens(text):
    # The chat template already starts with <|begin_of_text|>, so never add special tokens again
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def fit_prompt(row):
    """Returns (prompt, was_shrunk). Uses the whole abstract or a ±LOCAL_WINDOW-word window,
    narrowing the window around the acronym until the prompt fits PROMPT_BUDGET tokens."""
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    abbr = words[loc]
    words[loc] = f"[{abbr}]"
    window = max(loc, len(words) - loc) if CONTEXT_MODE == "full" else LOCAL_WINDOW
    shrunk = False
    while True:
        context_str = " ".join(words[max(0, loc - window): loc + window + 1])
        prompt = build_prompt(context_str, abbr)
        if n_tokens(prompt) <= PROMPT_BUDGET or window <= 5:
            return prompt, shrunk
        window, shrunk = window - 10, True

def encode_row(row):
    prompt, _ = fit_prompt(row)
    target = f"{str(row['LABEL']).strip().lower()}{tokenizer.eos_token}"
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target, truncation=True, max_length=TARGET_BUDGET, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + target_ids
    # Mask prompt tokens with -100 so loss is computed ONLY on the target expansion string
    return {"input_ids": input_ids, "labels": [-100] * len(prompt_ids) + target_ids, "attention_mask": [1] * len(input_ids)}

train_ds = Dataset.from_list([encode_row(r) for _, r in train_df.iterrows()])
dev_ds = Dataset.from_list([encode_row(r) for _, r in dev_df.iterrows()])

prompt_lens = [len(x) for x in train_ds["input_ids"]]
n_shrunk = sum(fit_prompt(r)[1] for d in (train_df, test_seen_df, test_unseen_df) for _, r in d.iterrows())
print(f"Train sequence tokens: median {np.median(prompt_lens):.0f} | max {max(prompt_lens)}")
print(f"Prompts whose context had to shrink to fit {PROMPT_BUDGET} tokens: {n_shrunk} / {len(train_df) + len(test_seen_df) + len(test_unseen_df)}")
print("\nExample prompt:\n" + fit_prompt(train_df.iloc[0])[0][-400:])

In [ ]:
# 7. Load 4-Bit Quantized Llama-3.2-1B-Instruct + Attach LoRA Adapters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 8. Train LoRA Adapter (checkpoints stay in Colab /content/, zero Drive writes)
schedule_args = dict(
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=EVAL_STEPS, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    prediction_loss_only=True,  # Otherwise eval keeps every dev logit (300 x ~1k tokens x 128k vocab) and runs out of memory
) if REGULARIZED else dict(save_strategy="no")

training_args = TrainingArguments(
    output_dir="/content/tmp_llama_lora_06",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=MAX_EPOCHS,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    weight_decay=WEIGHT_DECAY,
    neftune_noise_alpha=NEFTUNE_ALPHA,
    fp16=True,
    logging_steps=20,
    report_to="none",
    seed=SEED,
    **schedule_args,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds if REGULARIZED else None,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)] if REGULARIZED else [],
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_train = time.time()
trainer.train()
train_time = time.time() - start_train

if REGULARIZED:
    print(f"Stopped at step {trainer.state.global_step}; best dev-acronym loss {trainer.state.best_metric:.4f} "
          f"at {trainer.state.best_model_checkpoint}")

# Keep the adapter (~45 MB) so decoding can be re-run without retraining (Colab runtime only, not Drive)
model.save_pretrained(f"/content/adapter_06_{RUN_SLUG}")

In [ ]:
# 9. Decoding (greedy, beam, beam + initials re-ranking) + Metrics (same normalization and metrics as 04)
model.eval()
tokenizer.padding_side = "left"  # Required for batched causal generation

def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s-]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def token_f1(pred, truth):
    pred_tokens, truth_tokens = pred.split(), truth.split()
    if not pred_tokens or not truth_tokens:
        return float(pred_tokens == truth_tokens)
    common = sum((Counter(pred_tokens) & Counter(truth_tokens)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred_tokens), common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

def acronym_consistent(abbr, expansion):
    # Schwartz & Hearst (2003): every letter/digit of the acronym appears in the expansion in order
    # (scanning right to left), and the acronym's first character starts a word of the expansion.
    # PD -> "power doppler" passes; PD -> "peripheral nerve", CL -> "lymph" and RA -> "ra" fail.
    chars = [c for c in str(abbr).lower() if c.isalnum()]
    text = str(expansion).lower()
    if not chars or len(text.replace(" ", "")) < len(chars) + 2:
        return False  # Rejects echoes of the acronym itself ("ra") and stubs ("ecf"), which pass trivially
    j = len(text) - 1
    for i in range(len(chars) - 1, -1, -1):
        while j >= 0 and (text[j] != chars[i] or (i == 0 and j > 0 and text[j - 1].isalnum())):
            j -= 1
        if j < 0:
            return False
        j -= 1
    return True

def generate_candidates(prompts, num_beams, batch_size):
    # One list of candidates per prompt, best first (a single candidate when num_beams == 1)
    cands = []
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            inputs = tokenizer(
                prompts[i : i + batch_size],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,  # The chat template already starts with <|begin_of_text|>
            ).to(model.device)
            beam_args = dict(num_beams=num_beams, num_return_sequences=num_beams, early_stopping=True) if num_beams > 1 else {}
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                **beam_args,
            )
            input_len = inputs["input_ids"].shape[1]
            texts = [normalize_text(tokenizer.decode(seq[input_len:], skip_special_tokens=True)) for seq in outputs]
            cands.extend(texts[j : j + num_beams] for j in range(0, len(texts), num_beams))
    return cands

def metrics(preds, targets):
    return {
        "em": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro", zero_division=0),
        "tok_f1": np.mean([token_f1(p, t) for p, t in zip(preds, targets)]),
    }

def evaluate_split(df_split):
    prompts = [fit_prompt(row)[0] for _, row in df_split.iterrows()]
    acronyms = df_split["ACRONYM"].tolist()
    targets = [normalize_text(lbl) for lbl in df_split["LABEL"]]

    t0 = time.time()
    greedy = [c[0] for c in generate_candidates(prompts, 1, batch_size=16)]
    t_greedy = time.time() - t0

    t0 = time.time()
    beams = generate_candidates(prompts, NUM_BEAMS, batch_size=8)
    t_beam = time.time() - t0
    beam_top1 = [c[0] for c in beams]
    reranked = [next((c for c in cs if acronym_consistent(a, c)), cs[0]) for cs, a in zip(beams, acronyms)]

    return {
        "decoders": {
            "greedy": {**metrics(greedy, targets), "ms": t_greedy / len(prompts) * 1000, "preds": greedy},
            f"beam{NUM_BEAMS}": {**metrics(beam_top1, targets), "ms": t_beam / len(prompts) * 1000, "preds": beam_top1},
            f"beam{NUM_BEAMS}+initials": {**metrics(reranked, targets), "ms": t_beam / len(prompts) * 1000, "preds": reranked},
        },
        "targets": targets,
        "beams": beams,
        # Diagnostics: how often the gold answer is among the beams, and how often it passes the letter check at all
        "gold_in_beams": np.mean([t in cs for t, cs in zip(targets, beams)]),
        "gold_consistent": np.mean([acronym_consistent(a, t) for a, t in zip(acronyms, targets)]),
    }

seen = evaluate_split(test_seen_df)
unseen = evaluate_split(test_unseen_df)
dev = evaluate_split(dev_df)
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

print("\n" + "=" * 92)
print(f"Llama-3.2-1B + QLoRA (06) — {RUN_TAG}")
print("=" * 92)
print(f"Training Wall Time:   {train_time:.1f} sec")
print(f"Peak GPU VRAM:        {peak_vram_mb:.1f} MB")
print("-" * 92)
for split_name, r in [("Seen", seen), ("Unseen", unseen), ("Dev acronyms", dev)]:
    for dec_name, d in r["decoders"].items():
        print(f"{split_name:<13} {dec_name:<17} EM: {d['em']*100:6.2f}% | Macro F1: {d['macro_f1']*100:6.2f}% "
              f"| Token F1: {d['tok_f1']*100:6.2f}% | {d['ms']:6.1f} ms/sample")
    print(f"{split_name:<13} gold in top-{NUM_BEAMS} beams: {r['gold_in_beams']*100:.1f}% | "
          f"gold passes letter check: {r['gold_consistent']*100:.1f}%")
    print("-" * 92)

In [ ]:
# 10. Error Analysis — unseen EM per acronym, and sample predictions
best = f"beam{NUM_BEAMS}+initials"
unseen_out = test_unseen_df[["ACRONYM"]].copy()
unseen_out["true"] = unseen["targets"]
for dec_name, d in unseen["decoders"].items():
    unseen_out[dec_name] = d["preds"]
unseen_out["beams"] = [" | ".join(cs) for cs in unseen["beams"]]

print("Unseen EM by acronym:")
print(pd.DataFrame({dec: (unseen_out[dec] == unseen_out["true"]).groupby(unseen_out["ACRONYM"]).mean()
                    for dec in unseen["decoders"]})
      .assign(n=unseen_out.groupby("ACRONYM").size())
      .sort_values(best, ascending=False).to_string(float_format=lambda x: f"{x:.2f}"))

print(f"\nSample unseen predictions ({best}):")
for _, r in unseen_out.head(15).iterrows():
    print(f"  {'✓' if r[best] == r['true'] else '✗'} {r['ACRONYM']:<6} | Pred: {r[best]:<32} | True: {r['true']}")

unseen_out.to_csv(f"/content/unseen_predictions_06_{RUN_SLUG}.csv", index=False)  # Colab runtime only, not Drive